# 医学专业知识 RAG 系统 — 本地 LLM + PMC 数据源验证

本 Notebook 用于在 **Mac 本地 + 阿里云混合架构** 下完成「**验证本地语言模型 + 引入数据源**」这一阶段任务（详见 `任务.txt` 与 `schedule.md`）。

**本阶段目标（准备工作，不开发完整 RAG 系统）：**
- 验证本地 LLM（`deepseek-r1:7b` Q4_K_M）能在 Mac 上稳定运行
- 验证 Apple Silicon (M3) + Ollama 的推理可用性
- 接入 PubMed Central `oa_comm` 数据源（抽样下载、解析、字段抽取）
- 完成 Chroma 向量数据库最小可用性测试

**建议运行顺序：**

| 顺序 | 章节 | 说明 |
|------|------|------|
| 1 | **环境与硬件检查** | Python、MPS、conda env、依赖版本 |
| 2 | **路径与全局配置** | 项目根目录、数据目录、模型名、Ollama API |
| 3 | **Ollama 服务与模型管理** | 装 Ollama / 启服务 / 拉模型 |
| 4 | **本地 LLM 功能验证** | 跑测试问题、采集性能指标 |
| 5 | **PMC 数据源接入** | 抽样下载 / 解析 XML / 生成 jsonl |
| 6 | **Chroma 向量库 smoke test** | 最小可用性验证 |
| 7 | **LangChain 串联预览** | 仅展示组件接入方式，本阶段不实际入库 |
| 8 | **阶段验收汇总** | 自检 checklist + 文件结构 |

> **原计划平台分工**：
> - 章节 1~4、6~8 在 **Mac 本地** 完成
> - 章节 5 的下载/解析步骤主要在 **阿里云轻量应用服务器（海外节点）** 执行，最终样本同步回 Mac

---
## 每次重新打开 Notebook 必做（Kernel 重启 = 内存变量全部清空）

**请严格按下面顺序依次完成**：

| 顺序 | 操作 | 在哪做 | 说明 |
|------|------|------|------|
| 1 | 启动 **Jupyter Server** | Mac 终端 ① | `cd "01 验证模型" && ./start_jupyter.sh`；窗口保持不关 |
| 2 | Cursor 连接 server | Cursor | 首次：选择其他内核 → Existing Jupyter Server → 粘贴 token URL → 选 `Python (med-rag-verify)`；之后通常自动重连 |
| 3 | 启动 **Ollama 服务** | Mac 终端 ② | `cd "01 验证模型" && ./start_ollama.sh`；窗口保持不关，Ctrl+C 停止 |
| 4 | 运行 **章节 1 环境检查** | Notebook | 确认 Python / MPS / 依赖 / Conda 环境 |
| 5 | 运行 **章节 2 路径与全局配置** | Notebook | 定义 PROJECT_DIR / 模型名 / 缓存路径等 |

**两个终端窗口都要保持开着**：① Jupyter Server，② Ollama Service。

`start_jupyter.sh` 会自动把 `HF_HOME / TORCH_HOME / TRANSFORMERS_CACHE` 重定向到工程内 `caches/` 目录；`start_ollama.sh` 会把 `OLLAMA_MODELS` 指向工程内 `ollama_models/` 目录。**所有可能很大的下载都会留在工程内，方便备份与清理**。

若后续 cell 报：
- `name 'PROJECT_DIR' is not defined` → 步骤 5 未运行
- `connection refused` 连接 11434 → 步骤 3 未启动 Ollama

``` txt
工程容器化已经收尾。现在你可以测试新的启动流程：

在 Mac 终端关掉之前那个手动跑 jupyter notebook 的窗口（如果还在）
新开一个终端：
cd "/Users/sanxue/Desktop/work/实习/谷歌/01 验证模型"
./start_jupyter.sh
再开第二个终端：
cd "/Users/sanxue/Desktop/work/实习/谷歌/01 验证模型"
./start_ollama.sh
回 Cursor，把 kernel 重新指向新启的 Jupyter Server（URL 应该没变，可能自动重连）
跑 notebook cell 5，看输出里 HF_HOME / TORCH_HOME / TRANSFORMERS_CACHE 是否都指向 01 验证模型/caches/...
然后跑 cell 7（Ollama 服务检查）和 cell 9（医学问答测试）

---
## 1. 环境与硬件检查

运行后会打印：Python 版本、是否在 Apple Silicon、当前 conda 环境、磁盘可用、PyTorch / MPS 状态、9 个核心依赖版本。
若本步骤任一项失败，请回到 `schedule.md` 阶段 1 复检，再继续后续章节。

In [2]:
import os
import sys
import platform
import shutil

print("=== Python ===")
print(sys.version)
print("executable:", sys.executable)

print("\n=== 系统 ===")
print(f"platform : {platform.platform()}")
print(f"machine  : {platform.machine()}  (期望 arm64 = Apple Silicon)")

print("\n=== Conda 环境 ===")
print(f"CONDA_DEFAULT_ENV : {os.environ.get('CONDA_DEFAULT_ENV', '(未激活)')}")
print(f"CONDA_PREFIX      : {os.environ.get('CONDA_PREFIX', '(未激活)')}")
print("期望：med-rag-verify")

print("\n=== 磁盘 ===")
total, used, free = shutil.disk_usage("/")
print(f"总容量: {total/2**30:.2f} GB | 已用: {used/2**30:.2f} GB | 可用: {free/2**30:.2f} GB")

print("\n=== PyTorch 与设备能力 ===")
try:
    import torch
    print(f"torch          : {torch.__version__}")
    print(f"MPS available  : {torch.backends.mps.is_available()}")
    print(f"MPS built      : {torch.backends.mps.is_built()}")
    print(f"CUDA available : {torch.cuda.is_available()}  (Mac 无 NVIDIA GPU，预期 False)")
except Exception as e:
    print(f"  torch 检查失败: {e}")

print("\n=== 核心依赖版本 ===")
deps = ["langchain", "chromadb", "pandas", "datasets", "huggingface_hub",
        "requests", "tqdm", "lxml"]
for name in deps:
    try:
        mod = __import__(name)
        ver = getattr(mod, "__version__", "(no __version__)")
        print(f"  {name:18s} {ver}")
    except Exception as e:
        print(f"  {name:18s} 导入失败: {e}")

=== Python ===
3.11.15 (main, Mar 11 2026, 17:14:47) [Clang 20.1.8 ]
executable: /opt/miniconda3/envs/med-rag-verify/bin/python

=== 系统 ===
platform : macOS-26.4.1-arm64-arm-64bit
machine  : arm64  (期望 arm64 = Apple Silicon)

=== Conda 环境 ===
CONDA_DEFAULT_ENV : med-rag-verify
CONDA_PREFIX      : /opt/miniconda3/envs/med-rag-verify
期望：med-rag-verify

=== 磁盘 ===
总容量: 228.27 GB | 已用: 203.35 GB | 可用: 24.92 GB

=== PyTorch 与设备能力 ===
torch          : 2.11.0
MPS available  : True
MPS built      : True
CUDA available : False  (Mac 无 NVIDIA GPU，预期 False)

=== 核心依赖版本 ===
  langchain          1.2.18
  chromadb           1.5.9
  pandas             3.0.2
  datasets           4.8.5
  huggingface_hub    1.14.0
  requests           2.33.1
  tqdm               4.67.3
  lxml               6.1.0


---
## 2. 路径与全局配置

集中定义本任务的全部目录、模型名、API 端点、缓存重定向。
后续所有章节都会引用这里的变量。本单元会自动创建缺失的目录，并设置 HF / Torch / Transformers 缓存的环境变量。

**目录约定（相对本 Notebook 所在目录 `01 验证模型/`）：**

```
01 验证模型/
├── 任务.txt
├── schedule.md
├── requirements.txt
├── med-LLM-RAG.ipynb        # 本文件
├── start_ollama.sh          # 启动 Ollama 服务（指向工程内模型）
├── start_jupyter.sh         # 启动 Jupyter Server（指向工程内缓存）
├── ollama_models/           # Ollama 模型存储（4.4 GB）
│   ├── blobs/
│   └── manifests/
├── caches/                  # 运行时缓存重定向，HF / Torch / Transformers
│   ├── huggingface/         #   embedding 模型 + datasets
│   ├── torch/               #   torch hub
│   └── transformers/        #   transformers 库 model_cache
├── data/
│   ├── raw/                 # PMC 原始 tar.gz / 解压后的 XML（仅样本，全量在 OSS或硬盘）
│   └── processed/           # 解析后的 jsonl
├── outputs/
│   └── model_report.md      # 模型验证报告（手填）
└── chroma_db/               # Chroma 持久化目录（阶段 6 创建）
```

**核心设计**：所有「可能很大」的下载（Ollama 模型、HF embedding 模型、Torch 权重）都重定向到工程内子目录，Mac 上几乎不留残留，备份只需复制整个 `01 验证模型/` 即可。

In [3]:
import os

def _resolve_project_dir():
    """三道防线找工程根目录：
       1) 启动脚本里 export PROJECT_DIR
       2) 从 CWD 向上递归找含「任务.txt」的目录
       3) 都失败则抛错并提示如何修复
    """
    env_dir = os.environ.get("PROJECT_DIR")
    if env_dir and os.path.exists(os.path.join(env_dir, "任务.txt")):
        return env_dir, "环境变量 PROJECT_DIR"

    cur = os.path.abspath(os.getcwd())
    while True:
        if os.path.exists(os.path.join(cur, "任务.txt")):
            return cur, "向上搜索定位到 任务.txt"
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent

    raise RuntimeError(
        "未能定位工程根目录。请在终端用 ./start_jupyter.sh 重启 Jupyter Server，\n"
        "或确认 Notebook 所在目录里有 任务.txt 标志文件。"
    )

PROJECT_DIR, _src = _resolve_project_dir()
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"  (来源: {_src})")

DATA_DIR       = os.path.join(PROJECT_DIR, "data")
DATA_RAW_DIR   = os.path.join(DATA_DIR, "raw")
EXTRACT_DIR    = os.path.join(DATA_RAW_DIR, "extracted")
DATA_PROCESSED = os.path.join(DATA_DIR, "processed")
OUTPUTS_DIR    = os.path.join(PROJECT_DIR, "outputs")
CHROMA_DIR     = os.path.join(PROJECT_DIR, "chroma_db")
OLLAMA_MODELS_DIR = os.path.join(PROJECT_DIR, "ollama_models")

CACHES_DIR     = os.path.join(PROJECT_DIR, "caches")
HF_CACHE_DIR        = os.path.join(CACHES_DIR, "huggingface")
TORCH_CACHE_DIR     = os.path.join(CACHES_DIR, "torch")
TRANSFORMERS_CACHE_DIR = os.path.join(CACHES_DIR, "transformers")

for d in [DATA_RAW_DIR, EXTRACT_DIR, DATA_PROCESSED, OUTPUTS_DIR,
          HF_CACHE_DIR, TORCH_CACHE_DIR, TRANSFORMERS_CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

os.environ.setdefault("HF_HOME", HF_CACHE_DIR)
os.environ.setdefault("TORCH_HOME", TORCH_CACHE_DIR)
os.environ.setdefault("TRANSFORMERS_CACHE", TRANSFORMERS_CACHE_DIR)
os.environ.setdefault("HF_DATASETS_CACHE", os.path.join(HF_CACHE_DIR, "datasets"))

print("\n=== 工程内目录 ===")
for label, d in [("数据原始     ", DATA_RAW_DIR),
                 ("XML 解压目录 ", EXTRACT_DIR),
                 ("数据处理后   ", DATA_PROCESSED),
                 ("输出报告     ", OUTPUTS_DIR),
                 ("Chroma 持久化", CHROMA_DIR),
                 ("Ollama 模型 ", OLLAMA_MODELS_DIR),
                 ("HF 缓存      ", HF_CACHE_DIR),
                 ("Torch 缓存   ", TORCH_CACHE_DIR)]:
    exists = "存在" if os.path.exists(d) else "未创建"
    print(f"  {label}: {d}  [{exists}]")

OLLAMA_HOST  = "http://127.0.0.1:11434"
MODEL_NAME   = "deepseek-r1:7b"
MODEL_QUANT  = "Q4_K_M"

# PMC 数据源：本阶段只用 oa_comm/xml/ 下的 incr 增量小包做抽样验证。
# baseline 全量分片单个就有 43M~13G，30 GB 服务器盘装不下，本阶段无需用。
PMC_OA_BULK_URL    = "https://ftp.ncbi.nlm.nih.gov/pub/pmc/deprecated/oa_bulk/"
PMC_SUBSET         = "oa_comm"
PMC_XML_INDEX_URL  = f"{PMC_OA_BULK_URL}{PMC_SUBSET}/xml/"
SAMPLE_TARBALL     = "oa_comm_xml.incr.2026-02-10.tar.gz"   # ~5.9 MB，约 284 篇
SAMPLE_TARBALL_URL = f"{PMC_XML_INDEX_URL}{SAMPLE_TARBALL}"
SAMPLE_TARBALL_PATH = os.path.join(DATA_RAW_DIR, SAMPLE_TARBALL)
SAMPLE_JSONL       = os.path.join(DATA_PROCESSED, "sample.jsonl")
SAMPLE_LIMIT       = 100

# 阿里云 OSS：未来扩展用（本地小包验证够用时，本任务可不启用）
OSS_BUCKET = "med-rag-pmc"
OSS_REGION = "oss-ap-southeast-1"
OSS_RAW_PREFIX       = f"oss://{OSS_BUCKET}/raw/"
OSS_PROCESSED_PREFIX = f"oss://{OSS_BUCKET}/processed/"

print("\n=== 模型配置 ===")
print(f"OLLAMA_HOST : {OLLAMA_HOST}")
print(f"MODEL_NAME  : {MODEL_NAME}  (量化: {MODEL_QUANT})")

print("\n=== 数据源配置 ===")
print(f"PMC 索引页 : {PMC_XML_INDEX_URL}")
print(f"目标包     : {SAMPLE_TARBALL}")
print(f"目标包 URL : {SAMPLE_TARBALL_URL}")
print(f"本地落盘   : {SAMPLE_TARBALL_PATH}")
print(f"解压目录   : {EXTRACT_DIR}")
print(f"输出 jsonl : {SAMPLE_JSONL}")
print(f"采样上限   : {SAMPLE_LIMIT} 篇")

print("\n=== 阿里云 OSS（未来扩展，本地验证阶段不启用） ===")
print(f"Bucket    : {OSS_BUCKET}")
print(f"Region    : {OSS_REGION}")

print("\n=== 运行时缓存环境变量 (双保险) ===")
for k in ["HF_HOME", "TORCH_HOME", "TRANSFORMERS_CACHE", "HF_DATASETS_CACHE"]:
    print(f"  {k:20s} = {os.environ.get(k)}")

PROJECT_DIR: /Users/sanxue/Desktop/work/实习/谷歌/01 验证模型
  (来源: 向上搜索定位到 任务.txt)

=== 工程内目录 ===
  数据原始     : /Users/sanxue/Desktop/work/实习/谷歌/01 验证模型/data/raw  [存在]
  XML 解压目录 : /Users/sanxue/Desktop/work/实习/谷歌/01 验证模型/data/raw/extracted  [存在]
  数据处理后   : /Users/sanxue/Desktop/work/实习/谷歌/01 验证模型/data/processed  [存在]
  输出报告     : /Users/sanxue/Desktop/work/实习/谷歌/01 验证模型/outputs  [存在]
  Chroma 持久化: /Users/sanxue/Desktop/work/实习/谷歌/01 验证模型/chroma_db  [存在]
  Ollama 模型 : /Users/sanxue/Desktop/work/实习/谷歌/01 验证模型/ollama_models  [存在]
  HF 缓存      : /Users/sanxue/Desktop/work/实习/谷歌/01 验证模型/caches/huggingface  [存在]
  Torch 缓存   : /Users/sanxue/Desktop/work/实习/谷歌/01 验证模型/caches/torch  [存在]

=== 模型配置 ===
OLLAMA_HOST : http://127.0.0.1:11434
MODEL_NAME  : deepseek-r1:7b  (量化: Q4_K_M)

=== 数据源配置 ===
PMC 索引页 : https://ftp.ncbi.nlm.nih.gov/pub/pmc/deprecated/oa_bulk/oa_comm/xml/
目标包     : oa_comm_xml.incr.2026-02-10.tar.gz
目标包 URL : https://ftp.ncbi.nlm.nih.gov/pub/pmc/deprecated/oa_bulk/oa_comm/xml/oa_co

---
## 3. Ollama 服务与模型管理

**采用「按需启动 + 工程内模型管理」模式**：模型存在 `01 验证模型/ollama_models/`，通过环境变量 `OLLAMA_MODELS` 指向。这样工程整体可移植，也不会占用后台资源。

### 首次配置（已完成，仅记录）

```bash
brew install ollama
ollama pull deepseek-r1:7b
# 之后把 ~/.ollama/models 的内容迁移到工程目录 ollama_models/
# 启动脚本 start_ollama.sh 会自动设置 OLLAMA_MODELS
```

### 每次开工启动

在 Mac 终端执行（保持窗口不关）：

```bash
cd "/Users/sanxue/Desktop/work/实习/谷歌/01 验证模型"
./start_ollama.sh
```

启动成功标志：日志末尾出现
`Listening on 127.0.0.1:11434 (version 0.23.2)`

按 Ctrl+C 停止。

### 下方两个 cell

1. **服务可达性** — API 是否在 11434 端口响应
2. **模型清单** — `deepseek-r1:7b` 是否已加载

In [4]:
import requests

try:
    _ = OLLAMA_HOST, MODEL_NAME
except NameError:
    raise RuntimeError("环境未就绪：请先运行章节 2「路径与全局配置」。")

print(f"=== Ollama 服务检查: {OLLAMA_HOST} ===")
try:
    r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
    r.raise_for_status()
    data = r.json()
    print("服务状态: ONLINE")
    models = data.get("models", [])
    if not models:
        print("当前无已下载模型，请在终端执行: ollama pull deepseek-r1:7b")
    else:
        print(f"已下载模型 ({len(models)} 个):")
        for m in models:
            size_gb = m.get("size", 0) / (1024**3)
            print(f"  - {m.get('name'):30s} {size_gb:.2f} GB")
        names = [m.get("name", "") for m in models]
        if any(MODEL_NAME in n for n in names):
            print(f"\n[OK] 目标模型 '{MODEL_NAME}' 已就绪")
        else:
            print(f"\n[WARN] 目标模型 '{MODEL_NAME}' 未找到，请执行: ollama pull {MODEL_NAME}")
except requests.exceptions.ConnectionError:
    print("服务状态: OFFLINE")
    print("解决方法：在终端执行 `ollama serve`，或确认菜单栏 Ollama 图标已启动")
except Exception as e:
    print(f"检查失败: {type(e).__name__}: {e}")

=== Ollama 服务检查: http://127.0.0.1:11434 ===
服务状态: ONLINE
已下载模型 (1 个):
  - deepseek-r1:7b                 4.36 GB

[OK] 目标模型 'deepseek-r1:7b' 已就绪


---
## 4. 本地 LLM 功能验证

通过 Ollama 的 `/api/generate` REST 接口跑一组**中英文医学问题**，并采集每条问答的：

- 首 token 延迟（近似：API 总耗时）
- 平均 token/s
- 实际响应文本

测试问题覆盖：症状询问 / 风险因素 / 机制类 / 总结类，初步判断模型是否能用于医学领域问答。
本节不评分模型质量（不属于本阶段交付），仅做**能跑、能答、速度可接受**的验证。

第一版改进：
- 流式输出（stream=True）：实时看 token 喷出来，确认服务在干活
- 限制最大输出 token（num_predict=400）：避免无限思考
- 延长 timeout 到 600s：留足够余量

In [9]:
import time
import json
import sys
import requests

try:
    _ = OLLAMA_HOST, MODEL_NAME
except NameError:
    raise RuntimeError("环境未就绪：请先运行章节 2「路径与全局配置」。")

TEST_QUESTIONS = [
    "What are the common symptoms of type 2 diabetes?",
    "请简要解释高血压的主要风险因素。",
    "Summarize the relationship between chronic inflammation and cardiovascular disease.",
    "What is the mechanism of action of metformin?",
]

NUM_PREDICT = 400   # 限制最大生成 token：避免 deepseek-r1 思考无止境
TIMEOUT_S   = 600   # 总超时（含 prefill + 全部 decode）

def ollama_generate_stream(prompt: str, model: str = None, host: str = None,
                           num_predict: int = NUM_PREDICT, timeout: int = TIMEOUT_S,
                           preview_every: int = 40):
    """流式调用 Ollama /api/generate。
    - 实时打印 token 流（避免误以为卡死）
    - 返回 (text, elapsed_sec, eval_count, eval_duration_ns)
    """
    model = model or MODEL_NAME
    host = host or OLLAMA_HOST
    url = f"{host}/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": True,
        "options": {"num_predict": num_predict},
    }

    t0 = time.time()
    chunks = []
    eval_count = 0
    eval_duration = 0
    last_tick = 0

    with requests.post(url, json=payload, timeout=timeout, stream=True) as r:
        r.raise_for_status()
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            piece = obj.get("response", "")
            if piece:
                chunks.append(piece)
                eval_count += 1
                if eval_count - last_tick >= preview_every:
                    elapsed = time.time() - t0
                    tps = eval_count / elapsed if elapsed > 0 else 0
                    print(f"    .. {eval_count} tok  |  {elapsed:5.1f}s  |  {tps:.1f} tok/s", flush=True)
                    last_tick = eval_count
            if obj.get("done"):
                eval_count    = obj.get("eval_count", eval_count)
                eval_duration = obj.get("eval_duration", 0)
                break

    text = "".join(chunks)
    elapsed = time.time() - t0
    return text, elapsed, eval_count, eval_duration


results = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"\n----- Q{i}: {q} -----", flush=True)
    try:
        text, elapsed, n_tok, eval_ns = ollama_generate_stream(q)
        tps = (n_tok / (eval_ns / 1e9)) if eval_ns else (n_tok / elapsed if elapsed else 0.0)

        if "<think>" in text and "</think>" in text:
            visible = text.split("</think>", 1)[1].strip()
            think_chars = len(text) - len(visible)
        else:
            visible = text.strip()
            think_chars = 0

        preview = visible.replace("\n", " ")[:240]
        print(f"[耗时 {elapsed:.1f}s | 输出 {n_tok} tokens | {tps:.1f} tok/s | 思考占 {think_chars} 字]")
        print(f"答: {preview}{'...' if len(visible) > 240 else ''}")

        results.append({
            "q": q,
            "elapsed_s": round(elapsed, 2),
            "tokens": n_tok,
            "tok_per_s": round(tps, 2),
            "think_chars": think_chars,
            "answer_full": text,
            "answer_visible": visible,
        })
    except Exception as e:
        print(f"[FAIL] {type(e).__name__}: {e}")
        results.append({"q": q, "error": str(e)})

if results:
    ok = [r for r in results if "tok_per_s" in r]
    if ok:
        avg_tps = sum(r["tok_per_s"] for r in ok) / len(ok)
        avg_t   = sum(r["elapsed_s"] for r in ok) / len(ok)
        print(f"\n=== 性能小结 ===")
        print(f"成功 {len(ok)}/{len(TEST_QUESTIONS)} | 平均速度 {avg_tps:.1f} tok/s | 平均耗时 {avg_t:.1f}s")


----- Q1: What are the common symptoms of type 2 diabetes? -----
[耗时 272.0s | 输出 400 tokens | 1.6 tok/s | 思考占 0 字]
答: 

----- Q2: 请简要解释高血压的主要风险因素。 -----
[耗时 267.0s | 输出 400 tokens | 1.6 tok/s | 思考占 0 字]
答: 高血压的主要风险因素包括

----- Q3: Summarize the relationship between chronic inflammation and cardiovascular disease. -----
[耗时 245.5s | 输出 400 tokens | 1.7 tok/s | 思考占 0 字]
答: Chronic inflammation serves as a common underlying factor in various cardiovascular diseases, impacting multiple aspects of heart function:  1. **Circulation and Blood Pressure**: Chronic inflammation can lead to high blood

----- Q4: What is the mechanism of action of metformin? -----
[耗时 244.8s | 输出 400 tokens | 1.7 tok/s | 思考占 0 字]
答: 

=== 性能小结 ===
成功 4/4 | 平均速度 1.6 tok/s | 平均耗时 257.3s


第二版修正
- deepseek-r1:7b 会把思考过程放在单独字段：

{"response": "", "thinking": "Alright", "done": false}
而旧代码只收集了 response，没有收集 thinking。所以：

- Q1 / Q4 看起来空白：大概率 400 token 都耗在 thinking 里，真正回答还没开始。
- Q2 / Q3 只有开头：模型终于开始输出 response，但马上撞到 400 token 上限。
- 思考占 0 字 是旧代码统计错了：它只检查 response 里有没有 <think>...</think>，但 Ollama 实际把 thinking 分字段返回了。
- 平均 1.6 tok/s 是这次真实生成速度，偏慢但模型确实在跑；不是没量化，日志已确认 29/29 层都 offload 到 Metal，模型体积也是 Q4_K_M 的 4.4GB。

新版做了两件事：

- 设置 "think": False，关闭 deepseek-r1 的长思考链。
- 同时兼容收集 response 和 thinking，以后即使打开 thinking 也不会误判为空。

In [10]:
# 修正版：关闭 deepseek-r1 的 thinking，只验证直接回答能力
import time
import json
import requests

try:
    _ = OLLAMA_HOST, MODEL_NAME
except NameError:
    raise RuntimeError("环境未就绪：请先运行章节 2「路径与全局配置」。")

TEST_QUESTIONS = [
    "What are the common symptoms of type 2 diabetes?",
    "请简要解释高血压的主要风险因素。",
    "Summarize the relationship between chronic inflammation and cardiovascular disease.",
    "What is the mechanism of action of metformin?",
]

# 本阶段只验证「模型能否稳定回答」，不测试长链推理能力。
# deepseek-r1 默认会生成很长 thinking，Ollama 会把它放在独立的 thinking 字段；
# think=False 可关闭推理链，避免 num_predict 全耗在思考上。
THINK = False
NUM_PREDICT = 300
TIMEOUT_S = 300


def build_direct_prompt(question: str) -> str:
    return (
        "Answer the following medical question directly and concisely. "
        "Do not show reasoning steps. Keep the answer within 5 bullet points or one short paragraph.\n\n"
        f"Question: {question}"
    )


def ollama_generate_stream(prompt: str, model: str = None, host: str = None,
                           num_predict: int = NUM_PREDICT, timeout: int = TIMEOUT_S,
                           think: bool = THINK, preview_every: int = 50):
    """流式调用 Ollama /api/generate，分别收集 response 与 thinking。"""
    model = model or MODEL_NAME
    host = host or OLLAMA_HOST
    url = f"{host}/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": True,
        "think": think,
        "options": {"num_predict": num_predict},
    }

    t0 = time.time()
    response_chunks = []
    thinking_chunks = []
    approx_chunks = 0
    last_tick = 0
    final = {}

    with requests.post(url, json=payload, timeout=timeout, stream=True) as r:
        r.raise_for_status()
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            response_piece = obj.get("response", "")
            thinking_piece = obj.get("thinking", "")
            if response_piece:
                response_chunks.append(response_piece)
            if thinking_piece:
                thinking_chunks.append(thinking_piece)

            if response_piece or thinking_piece:
                approx_chunks += 1
                if approx_chunks - last_tick >= preview_every:
                    elapsed = time.time() - t0
                    print(f"    .. received {approx_chunks} chunks | {elapsed:5.1f}s", flush=True)
                    last_tick = approx_chunks

            if obj.get("done"):
                final = obj
                break

    elapsed = time.time() - t0
    response_text = "".join(response_chunks).strip()
    thinking_text = "".join(thinking_chunks).strip()
    eval_count = final.get("eval_count", approx_chunks)
    eval_duration = final.get("eval_duration", 0)
    done_reason = final.get("done_reason", "")
    return response_text, thinking_text, elapsed, eval_count, eval_duration, done_reason


results = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"\n----- Q{i}: {q} -----", flush=True)
    try:
        prompt = build_direct_prompt(q)
        answer, thinking, elapsed, n_tok, eval_ns, done_reason = ollama_generate_stream(prompt)
        tps = (n_tok / (eval_ns / 1e9)) if eval_ns else (n_tok / elapsed if elapsed else 0.0)
        hit_limit = (done_reason == "length") or (n_tok >= NUM_PREDICT)

        preview = answer.replace("\n", " ")[:260]
        print(
            f"[耗时 {elapsed:.1f}s | 输出 {n_tok} tokens | {tps:.1f} tok/s | "
            f"thinking {len(thinking)} 字 | done_reason={done_reason or 'unknown'}]"
        )
        print(f"答: {preview}{'...' if len(answer) > 260 else ''}")
        if hit_limit:
            print("[WARN] 本题达到 num_predict 上限，回答可能仍被截断。")

        results.append({
            "q": q,
            "elapsed_s": round(elapsed, 2),
            "tokens": n_tok,
            "tok_per_s": round(tps, 2),
            "done_reason": done_reason,
            "hit_num_predict_limit": hit_limit,
            "thinking_chars": len(thinking),
            "thinking": thinking,
            "answer_visible": answer,
        })
    except Exception as e:
        print(f"[FAIL] {type(e).__name__}: {e}")
        results.append({"q": q, "error": str(e)})

if results:
    ok = [r for r in results if "tok_per_s" in r]
    if ok:
        avg_tps = sum(r["tok_per_s"] for r in ok) / len(ok)
        avg_t = sum(r["elapsed_s"] for r in ok) / len(ok)
        truncated = sum(1 for r in ok if r.get("hit_num_predict_limit"))
        print("\n=== 性能小结 ===")
        print(f"成功 {len(ok)}/{len(TEST_QUESTIONS)} | 平均速度 {avg_tps:.1f} tok/s | 平均耗时 {avg_t:.1f}s | 截断 {truncated}/{len(ok)}")


----- Q1: What are the common symptoms of type 2 diabetes? -----
[耗时 33.9s | 输出 50 tokens | 1.7 tok/s | thinking 0 字 | done_reason=stop]
答: Common symptoms of type 2 diabetes include frequent urination, excessive thirst, increased urination even when dry mouthed, fatigue, weight loss, unexplained fatigue, numbness or tingling feet, and in severe cases, ketoacidosis.

----- Q2: 请简要解释高血压的主要风险因素。 -----
[耗时 24.7s | 输出 38 tokens | 1.7 tok/s | thinking 0 字 | done_reason=stop]
答: 高血压的主要风险因素包括遗传、生活方式（如饮食和运动）、家族史、超重或肥胖以及某些药物或生活习惯的变化，如长时间久坐、情绪紧张等。

----- Q3: Summarize the relationship between chronic inflammation and cardiovascular disease. -----
    .. received 50 chunks |  32.1s
[耗时 43.8s | 输出 69 tokens | 1.7 tok/s | thinking 0 字 | done_reason=stop]
答: Chronic inflammation is a key contributor to cardiovascular disease, as it leads to lipid accumulation in plaques on arterial walls, increases vascular wall thickness, and promotes the formation of atherosclerotic plaques. It also enhances bloo

In [11]:
try:
    _ = results, OUTPUTS_DIR
except NameError:
    raise RuntimeError("环境未就绪：请先运行上方「LLM 功能验证」单元生成 results。")

report_path = os.path.join(OUTPUTS_DIR, "model_test_results.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump({
        "model": MODEL_NAME,
        "quant": MODEL_QUANT,
        "num_predict": NUM_PREDICT,
        "timeout_s": TIMEOUT_S,
        "results": results,
    }, f, ensure_ascii=False, indent=2)
print(f"已保存详细问答 -> {report_path}")

已保存详细问答 -> /Users/sanxue/Desktop/work/实习/谷歌/01 验证模型/outputs/model_test_results.json


---
## 5. PMC `oa_comm` 数据源接入（**Mac 本地全流程**）

**数据源**：`https://ftp.ncbi.nlm.nih.gov/pub/pmc/deprecated/oa_bulk/oa_comm/xml/`

**本阶段策略**：先不开通阿里云服务器，直接在 Mac 上跑通整条链路。原因是 `oa_comm/xml/` 下有大量 `incr` 增量小包（最小约 5.9 MB），完全够 100 篇抽样验证；30 GB 服务器盘装不下 baseline 全量包（单包 43M~13G），但 Mac 本地装一个 5.9 MB 增量包完全无压力。

**本章节的 9 步**（每步独立可运行，逐 cell 留档）：

| 步 | 内容 | 产物 |
|---|---|---|
| 5.1 | 列出 `oa_comm/xml/` 所有 `.tar.gz`，按大小排序 | 候选清单 |
| 5.2 | 流式下载固定目标包 `oa_comm_xml.incr.2026-02-10.tar.gz` | `data/raw/*.tar.gz` |
| 5.3 | 解压 + 目录速览 | `data/raw/extracted/` |
| 5.4 | 打印一篇 XML 原文前若干行，直观感受 JATS 结构 | 控制台 |
| 5.5 | 定义 `parse_pmc_xml()` 字段抽取函数 | 函数对象 |
| 5.6 | 解析单篇验证字段（标题/摘要/正文长度） | 控制台 |
| 5.7 | 批量解析前 100 篇 → `sample.jsonl` | `data/processed/sample.jsonl` |
| 5.8 | pandas 字段完整性 + 长度分布 + 抽样预览 | 字段非空率表 |

> **未来扩展**：若后续需要 baseline 全量包（GB 级），再考虑开通阿里云海外节点 + OSS。当前阶段不启用。

### 5.1 列出 `oa_comm/xml/` 可用批次

抓取目录 HTML，过滤所有 `.tar.gz` 的文件名和大小，按大小升序展示。

**目的**：留档证明目标包 `oa_comm_xml.incr.2026-02-10.tar.gz` 存在、大小合理；同时也为未来选别的包提供候选清单。

**不做**：不下载，只列目录。

In [18]:
import re
import requests

try:
    _ = PMC_XML_INDEX_URL, SAMPLE_TARBALL
except NameError:
    raise RuntimeError("环境未就绪：请先运行章节 2「路径与全局配置」。")

print(f"GET {PMC_XML_INDEX_URL}")
r = requests.get(PMC_XML_INDEX_URL, timeout=60)
r.raise_for_status()
html = r.text

UNITS = {"K": 1024, "M": 1024**2, "G": 1024**3, "T": 1024**4}
def _to_bytes(sz: str) -> int:
    sz = sz.strip()
    if not sz or sz == "-":
        return 0
    m = re.fullmatch(r"([\d.]+)([KMGT])?", sz)
    if not m:
        return 0
    num, unit = m.groups()
    return int(float(num) * (UNITS.get(unit, 1)))

# NCBI 目录页是 Apache autoindex 风格：
#   <a href="oa_comm_xml.incr.2026-02-10.tar.gz">oa_comm_xml.incr.2026-02-10.tar.gz</a>  2026-02-10 01:08  5.9M
pattern = re.compile(
    r'<a href="([^"]+\.tar\.gz)">[^<]+</a>\s+([\d-]+\s+[\d:]+)\s+(\S+)'
)
rows = []
for href, ts, sz in pattern.findall(html):
    rows.append({"name": href, "modified": ts, "size_str": sz, "size_bytes": _to_bytes(sz)})

rows.sort(key=lambda x: x["size_bytes"])

print(f"\n共发现 {len(rows)} 个 .tar.gz")
print(f"\n=== 最小的 10 个（按大小升序） ===")
for r_ in rows[:10]:
    flag = "  ← 目标包" if r_["name"] == SAMPLE_TARBALL else ""
    print(f"  {r_['size_str']:>6s}  {r_['modified']}  {r_['name']}{flag}")

found = [r_ for r_ in rows if r_["name"] == SAMPLE_TARBALL]
if found:
    print(f"\n[OK] 目标包存在：{SAMPLE_TARBALL}  大小 {found[0]['size_str']}")
else:
    print(f"\n[WARN] 未在目录中找到目标包 {SAMPLE_TARBALL}（包名/日期可能已变）")

GET https://ftp.ncbi.nlm.nih.gov/pub/pmc/deprecated/oa_bulk/oa_comm/xml/

共发现 123 个 .tar.gz

=== 最小的 10 个（按大小升序） ===
    160K  2026-03-09 01:07  oa_comm_xml.incr.2026-03-09.tar.gz
    1.0M  2026-03-08 01:08  oa_comm_xml.incr.2026-03-08.tar.gz
    1.3M  2026-02-23 01:08  oa_comm_xml.incr.2026-02-23.tar.gz
    1.7M  2026-03-03 01:08  oa_comm_xml.incr.2026-03-03.tar.gz
    1.8M  2026-03-02 01:08  oa_comm_xml.incr.2026-03-02.tar.gz
    2.1M  2026-02-24 01:08  oa_comm_xml.incr.2026-02-24.tar.gz
    2.5M  2026-03-01 01:08  oa_comm_xml.incr.2026-03-01.tar.gz
    2.5M  2026-03-04 01:08  oa_comm_xml.incr.2026-03-04.tar.gz
    2.5M  2026-03-05 01:08  oa_comm_xml.incr.2026-03-05.tar.gz
    2.5M  2026-03-10 01:08  oa_comm_xml.incr.2026-03-10.tar.gz

[OK] 目标包存在：oa_comm_xml.incr.2026-02-10.tar.gz  大小 5.9M


### 5.2 下载目标包（Mac 本地直连）

用 `requests` 流式下载 `oa_comm_xml.incr.2026-02-10.tar.gz`（约 5.9 MB），落盘到 `data/raw/`。

**实现要点**：
- `stream=True` + `iter_content(chunk_size=1MB)`：避免一次性读到内存
- 每 1 MB 报一次进度，方便观察下载情况
- 下载完毕校验文件大小，确认完整
- 已存在且大小合理就跳过（idempotent）

In [19]:
import time
import requests

try:
    _ = SAMPLE_TARBALL_URL, SAMPLE_TARBALL_PATH
except NameError:
    raise RuntimeError("环境未就绪：请先运行章节 2「路径与全局配置」。")

CHUNK = 1024 * 1024  # 1 MiB

if os.path.exists(SAMPLE_TARBALL_PATH) and os.path.getsize(SAMPLE_TARBALL_PATH) > 1024 * 1024:
    print(f"[SKIP] 已存在 {SAMPLE_TARBALL_PATH}  ({os.path.getsize(SAMPLE_TARBALL_PATH) / 1024**2:.2f} MB)")
else:
    print(f"GET  {SAMPLE_TARBALL_URL}")
    print(f"  -> {SAMPLE_TARBALL_PATH}")
    t0 = time.time()
    with requests.get(SAMPLE_TARBALL_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))
        written = 0
        tmp_path = SAMPLE_TARBALL_PATH + ".part"
        with open(tmp_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=CHUNK):
                if not chunk:
                    continue
                f.write(chunk)
                written += len(chunk)
                pct = (written / total * 100) if total else 0
                print(f"  .. {written / 1024**2:6.2f} MB  ({pct:5.1f}%)", flush=True)
        os.replace(tmp_path, SAMPLE_TARBALL_PATH)
    elapsed = time.time() - t0
    size_mb = os.path.getsize(SAMPLE_TARBALL_PATH) / 1024**2
    speed = size_mb / elapsed if elapsed > 0 else 0
    print(f"\n[OK] 下载完成 {size_mb:.2f} MB，耗时 {elapsed:.1f}s，平均 {speed:.2f} MB/s")

GET  https://ftp.ncbi.nlm.nih.gov/pub/pmc/deprecated/oa_bulk/oa_comm/xml/oa_comm_xml.incr.2026-02-10.tar.gz
  -> /Users/sanxue/Desktop/work/实习/谷歌/01 验证模型/data/raw/oa_comm_xml.incr.2026-02-10.tar.gz
  ..   1.00 MB  ( 17.0%)
  ..   2.00 MB  ( 34.1%)
  ..   3.00 MB  ( 51.1%)
  ..   4.00 MB  ( 68.1%)
  ..   5.00 MB  ( 85.2%)
  ..   5.87 MB  (100.0%)

[OK] 下载完成 5.87 MB，耗时 48.1s，平均 0.12 MB/s


### 5.3 解压 tar.gz + 目录速览

用标准库 `tarfile` 把刚下好的包解压到 `data/raw/extracted/`，再汇报：

- 成员总数 / 解压后总大小
- 顶层有几个一级目录（PMC 按 `PMC008xxxxxx/` 这种段位分桶）
- 抽样 5 个 XML 相对路径，用来下一步 §5.4 直观感受 JATS

**幂等性**：如果 `extracted/` 下已有大量 `.xml`（说明上次解压过），就跳过解压步骤。

In [20]:
import glob
import tarfile
import time

try:
    _ = SAMPLE_TARBALL_PATH, EXTRACT_DIR
except NameError:
    raise RuntimeError("环境未就绪：请先运行章节 2「路径与全局配置」。")

existing = glob.glob(os.path.join(EXTRACT_DIR, "**/*.xml"), recursive=True)
if len(existing) >= 50:
    print(f"[SKIP] 已检测到 {len(existing)} 个 XML 文件，跳过解压")
else:
    print(f"OPEN {SAMPLE_TARBALL_PATH}")
    t0 = time.time()
    with tarfile.open(SAMPLE_TARBALL_PATH, "r:gz") as tf:
        members = tf.getmembers()
        print(f"  成员数 : {len(members)}")
        tf.extractall(EXTRACT_DIR)
    elapsed = time.time() - t0
    print(f"[OK] 解压完成，耗时 {elapsed:.1f}s")

xml_files = sorted(glob.glob(os.path.join(EXTRACT_DIR, "**/*.xml"), recursive=True))
total_bytes = sum(os.path.getsize(p) for p in xml_files)

top_dirs = sorted({os.path.relpath(p, EXTRACT_DIR).split(os.sep)[0] for p in xml_files})

print(f"\n=== 解压结果概览 ===")
print(f"XML 文件数  : {len(xml_files)}")
print(f"总大小      : {total_bytes / 1024**2:.2f} MB")
print(f"顶层目录数  : {len(top_dirs)}")
print(f"顶层目录    : {top_dirs[:10]}{' ...' if len(top_dirs) > 10 else ''}")
print(f"\n=== 抽样 5 个 XML 相对路径 ===")
for p in xml_files[:5]:
    print(f"  {os.path.relpath(p, EXTRACT_DIR)}  ({os.path.getsize(p)/1024:.1f} KB)")

OPEN /Users/sanxue/Desktop/work/实习/谷歌/01 验证模型/data/raw/oa_comm_xml.incr.2026-02-10.tar.gz
  成员数 : 284
[OK] 解压完成，耗时 0.2s

=== 解压结果概览 ===
XML 文件数  : 284
总大小      : 27.02 MB
顶层目录数  : 5
顶层目录    : ['PMC008xxxxxx', 'PMC009xxxxxx', 'PMC010xxxxxx', 'PMC011xxxxxx', 'PMC012xxxxxx']

=== 抽样 5 个 XML 相对路径 ===
  PMC008xxxxxx/PMC8774754.xml  (85.8 KB)
  PMC008xxxxxx/PMC8947183.xml  (93.3 KB)
  PMC009xxxxxx/PMC9962823.xml  (117.9 KB)
  PMC010xxxxxx/PMC10512598.xml  (49.3 KB)
  PMC010xxxxxx/PMC10560736.xml  (136.4 KB)


### 5.4 看一篇 XML 原文片段（认识 JATS 结构）

挑解压目录里第一篇 XML，打印**前 80 行**，让直观看到 PMC 用的 JATS（Journal Article Tag Suite）规范长什么样：`<article>`、`<front>`、`<article-id pub-id-type="pmc">`、`<article-title>`、`<abstract>`、`<body>`、`<p>` ……

这一步纯粹是「人看一眼」，不抽字段。下一节 §5.5 才用 lxml 写抽取规则。

In [21]:
import glob

xml_files = sorted(glob.glob(os.path.join(EXTRACT_DIR, "**/*.xml"), recursive=True))
if not xml_files:
    raise RuntimeError("EXTRACT_DIR 没有 XML，请先运行 §5.3 解压。")

first = xml_files[0]
print(f"FILE  {os.path.relpath(first, EXTRACT_DIR)}")
print(f"SIZE  {os.path.getsize(first) / 1024:.1f} KB")

PREVIEW_LINES = 80
with open(first, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if i > PREVIEW_LINES:
            print(f"... (省略，文件还有更多行)")
            break
        print(f"{i:4d}| {line.rstrip()}")

FILE  PMC008xxxxxx/PMC8774754.xml
SIZE  85.8 KB
   1| <!DOCTYPE article
   2| PUBLIC "-//NLM//DTD JATS (Z39.96) Journal Archiving and Interchange DTD with MathML3 v1.3 20210610//EN" "JATS-archivearticle1-3-mathml3.dtd">
   3| <article xmlns:mml="http://www.w3.org/1998/Math/MathML" xmlns:xlink="http://www.w3.org/1999/xlink" dtd-version="1.3" xml:lang="en" article-type="research-article"><?properties open_access?><processing-meta base-tagset="archiving" mathml-version="3.0" table-model="xhtml" tagset-family="jats"><restricted-by>pmc</restricted-by></processing-meta><front><journal-meta><journal-id journal-id-type="nlm-ta">Diagnostics (Basel)</journal-id><journal-id journal-id-type="iso-abbrev">Diagnostics (Basel)</journal-id><journal-id journal-id-type="publisher-id">diagnostics</journal-id><journal-title-group><journal-title>Diagnostics</journal-title></journal-title-group><issn pub-type="epub">2075-4418</issn><publisher><publisher-name>MDPI</publisher-name></publisher></journal-meta>
 

### 5.5 定义字段抽取函数 `parse_pmc_xml()`

针对 JATS 规范，用 `lxml.etree` + XPath 抽出 RAG 阶段最需要的核心字段：

| 字段 | XPath | 用途 |
|---|---|---|
| `pmcid` | `//article-id[@pub-id-type='pmc']` | 唯一标识 |
| `title` | `//article-title` | 检索 + 元数据 |
| `abstract` | `//abstract//p` | 短文本概览，可用于嵌入 |
| `body` | `//body//p` | 正文，RAG 检索主体 |
| `journal` | `//journal-title` | 元数据 |
| `pub_year` | `//pub-date/year` | 时间维度过滤 |

`_text(xpath)` 辅助函数把节点子树里的所有文本拼起来（处理 `<p>` 中夹杂的 `<italic>` 等标签）。

In [22]:
from lxml import etree

def parse_pmc_xml(xml_path: str) -> dict:
    """解析单个 PMC JATS XML，返回核心字段 dict。"""
    tree = etree.parse(xml_path)
    root = tree.getroot()

    def _text(xpath, sep=" "):
        nodes = root.xpath(xpath)
        return sep.join(("".join(n.itertext())).strip() for n in nodes if n is not None)

    pmcid    = _text("//article-id[@pub-id-type='pmc']")
    title    = _text("//article-title")
    abstract = _text("//abstract//p", sep="\n")
    body     = _text("//body//p", sep="\n")
    journal  = _text("//journal-title")
    pub_year = _text("//pub-date/year")

    return {
        "pmcid": pmcid,
        "title": title,
        "abstract": abstract,
        "body": body,
        "journal": journal,
        "pub_year": pub_year,
        "n_chars_abstract": len(abstract),
        "n_chars_body": len(body),
    }

print("[OK] parse_pmc_xml() 已定义，可被 §5.6 / §5.7 调用")

[OK] parse_pmc_xml() 已定义，可被 §5.6 / §5.7 调用


### 5.6 解析单篇验证字段抽取

挑解压目录第一篇 XML 喂给 `parse_pmc_xml()`，打印各字段长度 + 标题全文 + 摘要前 300 字 + 正文前 300 字，确认 §5.5 的抽取规则在真实数据上行得通。

In [23]:
import glob

try:
    _ = EXTRACT_DIR, parse_pmc_xml
except NameError:
    raise RuntimeError("环境未就绪：请先运行 §5.3 解压 与 §5.5 函数定义。")

xml_files = sorted(glob.glob(os.path.join(EXTRACT_DIR, "**/*.xml"), recursive=True))
if not xml_files:
    raise RuntimeError("EXTRACT_DIR 没有 XML，请先运行 §5.3 解压。")

first = xml_files[0]
rec = parse_pmc_xml(first)

print(f"FILE  {os.path.relpath(first, EXTRACT_DIR)}\n")

print("=== 字段长度 ===")
for k in ("pmcid", "title", "journal", "pub_year"):
    v = rec.get(k, "")
    print(f"  {k:9s}: {len(v):5d} 字  | {v}")
for k in ("abstract", "body"):
    v = rec.get(k, "")
    print(f"  {k:9s}: {len(v):5d} 字")

print("\n=== 摘要前 300 字 ===")
print(rec["abstract"][:300] + ("..." if len(rec["abstract"]) > 300 else ""))

print("\n=== 正文前 300 字 ===")
print(rec["body"][:300] + ("..." if len(rec["body"]) > 300 else ""))

FILE  PMC008xxxxxx/PMC8774754.xml

=== 字段长度 ===
  pmcid    :    10 字  | PMC8774754
  title    :  2973 字  | Axonemal Symmetry Break, a New Ultrastructural Diagnostic Tool for Primary Ciliary Dyskinesia? Primary Ciliary Dyskinesia Primary ciliary dyskinesia Laterality defects other than situs inversus totalis in primary ciliary dyskinesia: Insights into situs ambiguus and heterotaxy Motile ciliopathies European Respiratory Society guidelines for the diagnosis of primary ciliary dyskinesia Diagnosis of Primary Ciliary Dyskinesia. An Official American Thoracic Society Clinical Practice Guideline A human syndrome caused by immotile cilia Value of transmission electron microscopy for primary ciliary dyskinesia diagnosis in the era of molecular medicine: Genetic defects with normal and non-diagnostic ciliary ultrastructure Twenty-year review of quantitative transmission electron microscopy for the diagnosis of primary ciliary dyskinesia Mutations of DNAH11 in patients with primary ciliary dys

### 5.7 批量解析前 100 篇 → `sample.jsonl`

把 `EXTRACT_DIR` 里前 `SAMPLE_LIMIT` 篇 XML 全部喂给 `parse_pmc_xml()`，每篇一行写到 `data/processed/sample.jsonl`。失败的篇跳过并记录 skip 数，不让单点错误中断整批。

下一阶段的 chunking / embedding 就以这份 jsonl 为输入。

In [24]:
import json
import glob
import time
from tqdm import tqdm

try:
    _ = EXTRACT_DIR, SAMPLE_JSONL, SAMPLE_LIMIT, parse_pmc_xml
except NameError:
    raise RuntimeError("环境未就绪：请先运行 §5.3 解压 与 §5.5 函数定义。")

all_xml = sorted(glob.glob(os.path.join(EXTRACT_DIR, "**/*.xml"), recursive=True))
to_parse = all_xml[:SAMPLE_LIMIT]
print(f"解压目录共 {len(all_xml)} 篇 XML，本次解析前 {len(to_parse)} 篇 → {SAMPLE_JSONL}\n")

n_ok = 0
n_skip = 0
skip_reasons = []
t0 = time.time()

with open(SAMPLE_JSONL, "w", encoding="utf-8") as fout:
    for p in tqdm(to_parse, desc="parsing", unit="art"):
        try:
            rec = parse_pmc_xml(p)
            fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
            n_ok += 1
        except Exception as e:
            n_skip += 1
            skip_reasons.append(f"{os.path.basename(p)}: {type(e).__name__}: {e}")

elapsed = time.time() - t0
print(f"\n[OK] 写入 {n_ok} 条 / 跳过 {n_skip} 条，耗时 {elapsed:.2f}s")
print(f"     输出文件 : {SAMPLE_JSONL}")
print(f"     文件大小 : {os.path.getsize(SAMPLE_JSONL) / 1024:.1f} KB")
if skip_reasons:
    print("\n=== 跳过样本（前 5 条原因） ===")
    for line in skip_reasons[:5]:
        print(f"  - {line}")

解压目录共 284 篇 XML，本次解析前 100 篇 → /Users/sanxue/Desktop/work/实习/谷歌/01 验证模型/data/processed/sample.jsonl



parsing: 100%|█████████████████████████████████████████████| 100/100 [00:00<00:00, 340.90art/s]


[OK] 写入 100 条 / 跳过 0 条，耗时 0.33s
     输出文件 : /Users/sanxue/Desktop/work/实习/谷歌/01 验证模型/data/processed/sample.jsonl
     文件大小 : 4552.1 KB


### 5.8 pandas 字段完整性 + 长度分布 + 抽样预览

读回 `sample.jsonl`，给出：

- **字段非空率**：哪个字段经常空（医学文献偶有标题缺失 / 摘要缺失，提前发现）
- **长度分布**：`abstract` / `body` 字符数的分位数，判断 chunking 时 chunk_size 该怎么定
- **抽 3 条预览**：肉眼看一下数据样貌

这一步是阶段 3 的**验收**，跑通后 `schedule.md` §3.7 自动满足。

In [25]:
import pandas as pd

try:
    _ = SAMPLE_JSONL
except NameError:
    raise RuntimeError("环境未就绪：请先运行章节 2「路径与全局配置」。")

if not os.path.exists(SAMPLE_JSONL):
    raise RuntimeError(f"未找到 {SAMPLE_JSONL}，请先运行 §5.7。")

df = pd.read_json(SAMPLE_JSONL, lines=True)
print(f"共 {len(df)} 条记录\n")

print("=== 字段非空率 ===")
non_empty = (df.applymap(lambda v: bool(v)) if hasattr(df, "applymap")
             else df.map(lambda v: bool(v))).sum()
ratio = non_empty / len(df)
for col, n, r in zip(df.columns, non_empty, ratio):
    print(f"  {col:18s}: {int(n):4d}/{len(df)}  ({r*100:5.1f}%)")

print("\n=== 长度分布（字符数）===")
print(df[["n_chars_abstract", "n_chars_body"]].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9]).round(0))

print("\n=== 前 3 条预览 ===")
preview_cols = ["pmcid", "title", "journal", "pub_year",
                "n_chars_abstract", "n_chars_body"]
with pd.option_context("display.max_colwidth", 80, "display.width", 160):
    print(df[preview_cols].head(3).to_string(index=False))

print("\n=== 抽 1 条完整看一下 ===")
sample = df.iloc[0]
print(f"PMCID    : {sample['pmcid']}")
print(f"Title    : {sample['title']}")
print(f"Journal  : {sample['journal']}  ({sample['pub_year']})")
print(f"Abstract (前 200 字) :\n  {str(sample['abstract'])[:200]}...")
print(f"Body     (前 200 字) :\n  {str(sample['body'])[:200]}...")

共 100 条记录

=== 字段非空率 ===
  pmcid             :  100/100  (100.0%)
  title             :  100/100  (100.0%)
  abstract          :   97/100  ( 97.0%)
  body              :  100/100  (100.0%)
  journal           :  100/100  (100.0%)
  pub_year          :  100/100  (100.0%)
  n_chars_abstract  :   97/100  ( 97.0%)
  n_chars_body      :  100/100  (100.0%)

=== 长度分布（字符数）===
       n_chars_abstract  n_chars_body
count             100.0         100.0
mean             1708.0       39454.0
std               704.0       23959.0
min                 0.0         631.0
25%              1387.0       23904.0
50%              1628.0       31466.0
75%              1922.0       50563.0
90%              2325.0       66636.0
max              5181.0      128940.0

=== 前 3 条预览 ===
     pmcid                                                                                                                                                                                                                              

---
## 6. Chroma 向量数据库 smoke test

本阶段**不做真实入库**，仅验证：

- `chromadb` 可正常初始化
- 内存模式可插入/查询
- 持久化目录可写

后续 RAG 阶段会把 `sample.jsonl` 的文本分块、做 embedding、写入 Chroma；本阶段只完成「能用」验证。

In [26]:
import chromadb

try:
    _ = CHROMA_DIR
except NameError:
    raise RuntimeError("环境未就绪：请先运行章节 2「路径与全局配置」。")

print("=== 内存模式 smoke test ===")
client = chromadb.Client()
col = client.get_or_create_collection("smoke_test")
col.add(
    documents=[
        "Diabetes is a chronic disease characterized by elevated blood glucose.",
        "Hypertension is one of the most common cardiovascular risk factors.",
        "Metformin reduces hepatic glucose production via AMPK activation.",
    ],
    ids=["d1", "d2", "d3"],
)
res = col.query(query_texts=["What causes high blood pressure?"], n_results=2)
print("查询命中:")
for doc, _id in zip(res["documents"][0], res["ids"][0]):
    print(f"  - [{_id}] {doc}")

print(f"\n=== 持久化模式 smoke test (path={CHROMA_DIR}) ===")
pc = chromadb.PersistentClient(path=CHROMA_DIR)
pcol = pc.get_or_create_collection("med_smoke")
pcol.add(documents=["persistent diabetes test"], ids=["p1"])
print("持久化插入成功，集合数:", pcol.count())
print("Chroma 持久化目录已建立 ->", CHROMA_DIR)

=== 内存模式 smoke test ===


/Users/sanxue/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|█| 79.3M/79.3M [00:2


查询命中:
  - [d2] Hypertension is one of the most common cardiovascular risk factors.
  - [d1] Diabetes is a chronic disease characterized by elevated blood glucose.

=== 持久化模式 smoke test (path=/Users/sanxue/Desktop/work/实习/谷歌/01 验证模型/chroma_db) ===
持久化插入成功，集合数: 1
Chroma 持久化目录已建立 -> /Users/sanxue/Desktop/work/实习/谷歌/01 验证模型/chroma_db


---
## 7. LangChain 串联预览（占位 / 仅展示概念）

> 本阶段**不实际执行**该 cell，只用于理解后续 RAG 工程会如何把组件串起来。
> 真正的 chunking / embedding / retrieval / chain 都放到下一阶段。

预览流程：

```
PMC sample.jsonl
   ↓ TextLoader / 自定义 Loader
chunked Documents
   ↓ HuggingFaceEmbeddings (MPS 加速) 或 OllamaEmbeddings
向量
   ↓ Chroma.from_documents(...)
向量库
   ↓ retriever
Retrieval QA chain (LLM=ChatOllama(deepseek-r1:7b))
   ↓
医学问答输出
```

In [ ]:
# [占位 / 不在本阶段执行] —— 仅展示下一阶段的代码雏形
RAG_PREVIEW = """
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain.chains import RetrievalQA

# 1) 加载并分块
loader = JSONLoader(file_path=SAMPLE_JSONL, jq_schema=".body", text_content=True)
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=80)
docs = splitter.split_documents(loader.load())

# 2) Embedding（CPU/MPS 都可）
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                            model_kwargs={"device": "mps"})

# 3) 向量库
db = Chroma.from_documents(docs, embedding=emb, persist_directory=CHROMA_DIR)

# 4) LLM + RAG 链
llm = ChatOllama(model=MODEL_NAME, base_url=OLLAMA_HOST)
qa = RetrievalQA.from_chain_type(llm=llm, retriever=db.as_retriever(k=4))

print(qa.invoke({"query": "What is the typical first-line treatment for type 2 diabetes?"}))
"""
print(RAG_PREVIEW)

---
## 8. 阶段验收汇总

读取各步骤的产物状态，给出一个一目了然的勾选清单，对应 `schedule.md` 的 5.2 自检 Checklist。

In [27]:
try:
    _ = PROJECT_DIR, OLLAMA_HOST, MODEL_NAME, SAMPLE_JSONL, CHROMA_DIR
except NameError:
    raise RuntimeError("环境未就绪：请先运行章节 2「路径与全局配置」。")

def _flag(ok: bool) -> str:
    return "[x]" if ok else "[ ]"

print("=== 阶段验收 Checklist ===\n")

req = os.path.join(PROJECT_DIR, "requirements.txt")
print(f"{_flag(os.path.exists(req))} requirements.txt 已生成")

try:
    r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=3)
    models = [m.get("name", "") for m in r.json().get("models", [])]
    print(f"{_flag(r.ok)} Ollama 服务在线")
    print(f"{_flag(any(MODEL_NAME in n for n in models))} 目标模型 {MODEL_NAME} 已下载")
except Exception:
    print("[ ] Ollama 服务在线")
    print(f"[ ] 目标模型 {MODEL_NAME} 已下载")

report = os.path.join(OUTPUTS_DIR, "model_test_results.json")
print(f"{_flag(os.path.exists(report))} 模型测试结果已落盘 (outputs/model_test_results.json)")

print(f"{_flag(os.path.exists(SAMPLE_JSONL))} PMC sample.jsonl 已生成")
if os.path.exists(SAMPLE_JSONL):
    try:
        n_lines = sum(1 for _ in open(SAMPLE_JSONL, "r", encoding="utf-8"))
        print(f"      └─ 样本数: {n_lines} 篇  (目标 ≥ 100)")
    except Exception:
        pass

print(f"{_flag(os.path.isdir(CHROMA_DIR))} Chroma 持久化目录已建立")

print("\n=== 当前项目文件结构 ===")
for root, dirs, files in os.walk(PROJECT_DIR):
    if any(x in root for x in [".ipynb_checkpoints", "__pycache__", "chroma_db"]):
        continue
    level = root.replace(PROJECT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in sorted(files):
            print(f"{indent}  {f}")

=== 阶段验收 Checklist ===

[x] requirements.txt 已生成
[x] Ollama 服务在线
[x] 目标模型 deepseek-r1:7b 已下载
[x] 模型测试结果已落盘 (outputs/model_test_results.json)
[x] PMC sample.jsonl 已生成
      └─ 样本数: 100 篇  (目标 ≥ 100)
[x] Chroma 持久化目录已建立

=== 当前项目文件结构 ===
01 验证模型/
  med-LLM-RAG.ipynb
  requirements.txt
  schedule.md
  start_jupyter.sh
  start_ollama.sh
  任务.txt
  ollama_models/
    blobs/
    manifests/
      registry.ollama.ai/
        library/
          deepseek-r1/
  data/
    processed/
    raw/
      extracted/
        PMC008xxxxxx/
        PMC011xxxxxx/
        PMC009xxxxxx/
        PMC010xxxxxx/
        PMC012xxxxxx/
  outputs/
    model_test_results.json
  caches/
    torch/
    transformers/
    huggingface/
      datasets/


---
## 附录

### A. 项目文件作用一览

| 文件 / 目录 | 作用 |
|---|---|
| `任务.txt` | 实习导师下发的原始任务说明 |
| `schedule.md` | 阶段计划表（学习 + 执行进度） |
| `requirements.txt` | Python 依赖版本锁定，复现环境用 |
| `med-LLM-RAG.ipynb` | **本文件**：所有验证操作的统一入口与说明 |
| `data/raw/` | PMC 原始 `.tar.gz` 与解压后的 XML（仅样本） |
| `data/processed/sample.jsonl` | 解析后的医学文献样本，下一阶段 RAG 的输入 |
| `outputs/model_test_results.json` | 模型测试问答与速度指标 |
| `outputs/model_report.md` | （手填）模型选型与性能小结 |
| `chroma_db/` | Chroma 持久化目录（阶段 6 创建） |

### B. 常见问题

- **`connection refused` 连接 Ollama**：终端 `ollama serve`，或确认 macOS 菜单栏 Ollama 图标已启动。
- **`ollama pull` 卡住**：换网络环境；或先 `OLLAMA_HOST=...` 设置代理；备选 Hugging Face GGUF 镜像 + `ollama create -f Modelfile`。
- **kernel 找不到 `torch`**：Notebook 顶部选 kernel 时确认是 `med-rag-verify`，不是 `base`。
- **MPS 推理变慢**：M3 在长上下文上可能内存压力大；调小 `num_ctx` 或换更激进量化（Q3_K_M）。
- **PMC XML 解析失败**：极少数文档 DTD 损坏，外层用 try/except 跳过即可，不影响整体抽样。

### C. 学习资源

- Ollama 文档：https://github.com/ollama/ollama
- LangChain RAG：https://python.langchain.com/docs/tutorials/rag/
- Chroma 文档：https://docs.trychroma.com/
- PMC OA Bulk：https://www.ncbi.nlm.nih.gov/pmc/tools/openftlist/
- JATS XML 规范：https://jats.nlm.nih.gov/